In [ ]:
# ──── 1. 라이브러리 import ────────────────────────────────────────
import os
import torch
import torch.nn as nn
import torch.optim as optim
import torchvision
import torchvision.transforms as transforms
from torch.utils.data import DataLoader, random_split
from torch.utils.tensorboard import SummaryWriter
from tqdm import trange

# ──── 2. VGG 블록 & 모델 정의 ────────────────────────────────────
def conv_2_block(in_dim, out_dim):
    return nn.Sequential(
        nn.Conv2d(in_dim, out_dim, kernel_size=3, padding=1),
        nn.ReLU(inplace=True),
        nn.Conv2d(out_dim, out_dim, kernel_size=3, padding=1),
        nn.ReLU(inplace=True),
        nn.MaxPool2d(2, 2)
    )

def conv_3_block(in_dim, out_dim):
    return nn.Sequential(
        nn.Conv2d(in_dim, out_dim, 3, padding=1),
        nn.ReLU(inplace=True),
        nn.Conv2d(out_dim, out_dim, 3, padding=1),
        nn.ReLU(inplace=True),
        nn.Conv2d(out_dim, out_dim, 3, padding=1),
        nn.ReLU(inplace=True),
        nn.MaxPool2d(2, 2)
    )

class VGG16_CIFAR(nn.Module):
    def __init__(self, base_dim=64, num_classes=10):
        super().__init__()
        self.features = nn.Sequential(
            conv_2_block(3, base_dim),
            conv_2_block(base_dim, base_dim*2),
            conv_3_block(base_dim*2, base_dim*4),
            conv_3_block(base_dim*4, base_dim*8),
            conv_3_block(base_dim*8, base_dim*8),
        )
        # FC 레이어 첫 두 개에 Dropout(0.5) 적용
        self.classifier = nn.Sequential(
            nn.Linear(base_dim*8*1*1, 4096),
            nn.ReLU(inplace=True),
            nn.Dropout(0.5),
            nn.Linear(4096, 4096),
            nn.ReLU(inplace=True),
            nn.Dropout(0.5),
            nn.Linear(4096, num_classes),
        )

    def forward(self, x):
        x = self.features(x)
        x = x.view(x.size(0), -1)
        return self.classifier(x)

# ──── 3. 설정 ───────────────────────────────────────────────────
device        = torch.device("cuda" if torch.cuda.is_available() else "cpu")
batch_size    = 64      # mini-batch size
learning_rate = 0.01    # 초기 LR
num_epochs    = 20

# ──── 4. 데이터 준비 & 분할 ───────────────────────────────────────
transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.4914,0.4822,0.4465),
                         (0.2470,0.2435,0.2616))
])

# 전체 훈련셋 불러오기
full_train_ds = torchvision.datasets.CIFAR10(
    root="./data", train=True, download=True, transform=transform)

# 45k / 5k 로 train/val 분할
torch.manual_seed(42)
train_ds, val_ds = random_split(full_train_ds, [45000, 5000])

# 별도로 test셋 불러오기
test_ds = torchvision.datasets.CIFAR10(
    root="./data", train=False, download=True, transform=transform)

train_loader = DataLoader(train_ds, batch_size=batch_size, shuffle=True,  num_workers=4)
val_loader   = DataLoader(val_ds,   batch_size=batch_size, shuffle=False, num_workers=4)
test_loader  = DataLoader(test_ds,  batch_size=batch_size, shuffle=False, num_workers=4)

# ──── 5. 모델·손실·옵티마이저·TensorBoard 준비 ─────────────────
model     = VGG16_CIFAR(base_dim=64, num_classes=10).to(device)
criterion = nn.CrossEntropyLoss()  
optimizer = optim.SGD(
    model.parameters(),
    lr=learning_rate,
    momentum=0.9,
    weight_decay=5e-4
)
writer    = SummaryWriter(log_dir="runs/cifar10_vgg")

# ──── 6. 학습 루프 ─────────────────────────────────────────────
global_step = 0
for epoch in range(1, num_epochs+1):
    model.train()
    running_loss = 0.0

    for images, labels in train_loader:
        images, labels = images.to(device), labels.to(device)

        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        running_loss += loss.item()
        global_step += 1
        if global_step % 50 == 0:
            writer.add_scalar("Train/Batch_Loss", loss.item(), global_step)

    epoch_loss = running_loss / len(train_loader)
    writer.add_scalar("Train/Epoch_Loss", epoch_loss, epoch)

    # ──── 7. 검증(Validation) ────────────────────────────────────
    model.eval()
    val_loss = 0.0
    correct = 0
    total   = 0
    with torch.no_grad():
        for images, labels in val_loader:
            images, labels = images.to(device), labels.to(device)
            outputs = model(images)
            loss = criterion(outputs, labels)
            val_loss += loss.item()
            _, preds = outputs.max(1)
            correct += (preds == labels).sum().item()
            total   += labels.size(0)

    val_loss /= len(val_loader)
    val_acc   = correct / total
    writer.add_scalar("Val/Loss", val_loss, epoch)
    writer.add_scalar("Val/Acc",  val_acc,   epoch)

    print(f"[Epoch {epoch:02d}/{num_epochs}] "
          f"Train Loss: {epoch_loss:.4f}  "
          f"Val Loss: {val_loss:.4f}  Val Acc: {val_acc:.4f}")

# ──── 8. 최종 테스트셋 평가 ─────────────────────────────────────
model.eval()
test_loss = 0.0
correct = 0
total   = 0
with torch.no_grad():
    for images, labels in test_loader:
        images, labels = images.to(device), labels.to(device)
        outputs = model(images)
        loss = criterion(outputs, labels)
        test_loss += loss.item()
        _, preds = outputs.max(1)
        correct += (preds == labels).sum().item()
        total   += labels.size(0)

test_loss /= len(test_loader)
test_acc   = correct / total
writer.add_scalar("Test/Loss", test_loss)
writer.add_scalar("Test/Acc",  test_acc)

print(f"\n🎯 Final Test Loss: {test_loss:.4f}  Test Acc: {test_acc:.4f}")

# ──── 9. 마무리 ────────────────────────────────────────────────
writer.close()
